# V14 — Explanations, lung- and lesion-level grounding, causal occlusion (study design v2, §5–6)
**GPU, about 3–5 h, inference only.** Inputs: V12 output + the four V13 outputs (one per architecture).

| Part | Models | Images | What |
|---|---|---|---|
| A | 12 seed-42 models | grounding sample: per cohort 200 positives with boxes + 100 negatives (internal, NIH: 300) | occlusion-sensitivity maps (primary), Grad-CAM, attention rollout (ViT) → LFR, LeFR, LeFR-in-lung, pointing game, box AUROC; aggregate maps |
| B | all 36 models | 100 images per cohort | Grad-CAM metrics → seed variability of grounding |
| C | 12 seed-42 models | full cohort samples | region occlusion: lungs, non-lungs, v1 geometric bands, lesion box, area-matched random controls → per-image probabilities |

Statistics (bootstrap, TOST, mixed models) are run offline from the saved per-image files.

**Lung definition:** the dilated lung mask in every arm's frame (the region arm C can see), as stored by V12 (2026-10-17 version).
Resume-safe: each model writes its own files and finished models are skipped.

In [ ]:
import os, json, glob, time, math, random, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import torch, torch.nn.functional as F
import timm
from sklearn.metrics import roc_auc_score
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 42
INPUT = os.environ.get("V14_INPUT", "/kaggle/input")
OUT = os.environ.get("V14_OUT", "/kaggle/working")
QUICK_TEST = os.environ.get("QUICK_TEST", "0") == "1"
ARCHS = ["vit_base_patch16_224", "densenet121", "convnext_tiny", "swin_tiny_patch4_window7_224"]
ARMS, SEEDS = ["raw", "roi", "lung"], [42, 1337, 2024]
N_POS, N_NEG, N_NOBOX, N_SEEDVAR = (3, 2, 4, 3) if QUICK_TEST else (200, 100, 300, 100)
GRID = 4 if QUICK_TEST else 14                              # occlusion-sensitivity grid (16-px patches at 14)
TAUS, POINT_TOL = (0.0, 0.1, 0.2, 0.4), 8
OCC_BATCH = 16 if QUICK_TEST else 98
S, MEAN, STD, FILL = 224, 0.4769, 0.2414, 122
for d in ("grounding", "occlusion", "maps"): os.makedirs(os.path.join(OUT, d), exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
T0 = time.time()

def find_file(name):
    for r, _, fs in os.walk(INPUT):
        if name in fs: return os.path.join(r, name)
    return None

## 1. Cache, samples and checkpoints

In [ ]:
MANIFEST = find_file("manifest_v2.csv"); assert MANIFEST, "add the V12 output"
CDIR = os.path.dirname(MANIFEST)
meta = pd.read_csv(MANIFEST)
cache = {}
for c in meta.cohort.unique():
    z = np.load(os.path.join(CDIR, f"cache_{c}.npz"))
    cache[c] = {k: z[k] for k in z.files}
chance = pd.read_csv(os.path.join(CDIR, "chance_levels_v2.csv"))
EVAL = {"internal": meta[(meta.cohort == "internal") & (meta.split == "test")]}
for c in ("rsna", "nih", "vindr"):
    if c in cache: EVAL[c] = meta[meta.cohort == c]
if QUICK_TEST:
    EVAL = {c: d.groupby("label", group_keys=False).head(5) for c, d in EVAL.items()}

def grounding_sample(d):
    r = np.random.default_rng(SEED)
    pos_box = d[(d.label == 1) & (d.n_boxes > 0)]
    if len(pos_box):
        p = pos_box.iloc[r.permutation(len(pos_box))[:N_POS]]
        neg = d[d.label == 0]; n = neg.iloc[r.permutation(len(neg))[:N_NEG]]
        return pd.concat([p, n])
    half = N_NOBOX // 2
    return pd.concat([d[d.label == k].iloc[r.permutation(int((d.label == k).sum()))[:half]] for k in (1, 0)])
GSAMPLE = {c: grounding_sample(d) for c, d in EVAL.items()}
SEEDVAR = {c: g.iloc[np.random.default_rng(SEED + 1).permutation(len(g))[:N_SEEDVAR]] for c, g in GSAMPLE.items()}
print({c: (len(g), int(g.label.sum())) for c, g in GSAMPLE.items()})

CKPT = {os.path.basename(p)[:-4]: p for p in glob.glob(os.path.join(INPUT, "**", "ckpt", "*.pth"), recursive=True)}
print(len(CKPT), "checkpoints found")

def load_model(arch, arm, seed):
    key = f"{arch}__{arm}__s{seed}"
    if key not in CKPT: return None
    m = timm.create_model(arch, pretrained=False, num_classes=2)
    m.load_state_dict(torch.load(CKPT[key], map_location="cpu")["state_dict"])
    if arch.startswith("vit"):
        for b in m.blocks: b.attn.fused_attn = False           # expose attention weights for rollout
    return m.to(device).eval()

def lung_def(arm, m):
    """One lung definition in every frame: the dilated lung mask (the region arm C can see). From the 2026-10-17 V12 onwards the
    stored masks are already dilated in all three arms, so this is the identity; kept as the single place to change it."""
    return m.astype(np.uint8)

def to_tensor(u8):                                            # (N,224,224) uint8 -> (N,3,224,224) normalised
    x = torch.from_numpy(np.ascontiguousarray(u8)).float().div(255.0).sub(MEAN).div(STD)
    return x.unsqueeze(1).expand(-1, 3, -1, -1).contiguous()

@torch.no_grad()
def probs(model, u8, bs=128):
    out = []
    for i in range(0, len(u8), bs):
        with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            out.append(torch.softmax(model(to_tensor(u8[i:i + bs]).to(device)).float(), 1).cpu())
    return torch.cat(out).numpy()

## 2. Explanation methods

In [ ]:
@torch.no_grad()
def occlusion_map(model, img_u8):
    """Model-agnostic: drop in predicted-class probability when each 16-px patch is set to mean grey."""
    p0 = probs(model, img_u8[None])[0]; c = int(p0.argmax())
    ps = S // GRID; variants = []
    for gy in range(GRID):
        for gx in range(GRID):
            v = img_u8.copy(); v[gy * ps:(gy + 1) * ps, gx * ps:(gx + 1) * ps] = FILL; variants.append(v)
    pv = probs(model, np.stack(variants), bs=OCC_BATCH)[:, c]
    m = np.clip(p0[c] - pv, 0, None).reshape(GRID, GRID).astype(np.float32)
    return cv2.resize(m, (S, S), interpolation=cv2.INTER_LINEAR), c, float(p0[1])

def cam_target(model, arch):
    if arch.startswith("vit"): return model.blocks[-1].norm1
    if arch.startswith("swin"): return model.layers[-1]
    if arch.startswith("densenet"): return model.features
    if arch.startswith("convnext"): return model.stages[-1]
    raise ValueError(arch)

def to_nchw(a, model):
    if a.dim() == 3:                                           # tokens (B, N, C)
        a = a[:, getattr(model, "num_prefix_tokens", 1):, :]
        n = int(round(math.sqrt(a.shape[1]))); return a.transpose(1, 2).reshape(a.shape[0], -1, n, n)
    if a.dim() == 4 and a.shape[-1] == getattr(model, "num_features", -1) and a.shape[1] != a.shape[-1]:
        return a.permute(0, 3, 1, 2)                           # NHWC (timm Swin)
    return a

def gradcam(model, arch, img_u8):
    store = {}
    layer = cam_target(model, arch)
    h1 = layer.register_forward_hook(lambda m, i, o: store.__setitem__("a", o))
    x = to_tensor(img_u8[None]).to(device)
    with torch.enable_grad():
        logits = model(x); c = int(logits.argmax(1))
        a = store["a"]; a.retain_grad(); model.zero_grad(); logits[0, c].backward()
        A, G = to_nchw(a, model), to_nchw(a.grad, model)
    h1.remove()
    w = G.mean(dim=(2, 3), keepdim=True)
    cam = F.relu((w * A).sum(1))[0].detach().float().cpu().numpy()
    return cv2.resize(cam, (S, S), interpolation=cv2.INTER_LINEAR), c

def rollout(model, img_u8):
    mats = []
    hooks = [b.attn.attn_drop.register_forward_hook(lambda m, i, o: mats.append(o.detach().float().cpu())) for b in model.blocks]
    with torch.no_grad(): model(to_tensor(img_u8[None]).to(device))
    for h in hooks: h.remove()
    if not mats: return None
    N = mats[0].shape[-1]; R = torch.eye(N)
    for a in mats:
        a = a[0].mean(0) + torch.eye(N); a = a / a.sum(-1, keepdim=True); R = a @ R
    m = R[0, model.num_prefix_tokens:].numpy(); n = int(round(math.sqrt(m.size)))
    return cv2.resize(m.reshape(n, n).astype(np.float32), (S, S), interpolation=cv2.INTER_LINEAR)

## 3. Grounding metrics (design §5)

In [ ]:
def norm01(s):
    s = s - s.min(); return s / s.max() if s.max() > 1e-12 else s

def metrics(sal, lung, box, tau):
    s = norm01(sal); L = lung.astype(bool); B = box.astype(bool) if box is not None else None
    w = s * (s > tau)
    tot = w.sum(); out = dict(lfr=float((w * L).sum() / tot) if tot > 0 else np.nan)
    if B is not None and B.any():
        wl = (w * L).sum()
        out["lefr"] = float((w * B).sum() / tot) if tot > 0 else np.nan
        out["lefr_in_lung"] = float((w * B * L).sum() / wl) if wl > 0 else np.nan
        if tau == 0.2:
            iy, ix = np.unravel_index(np.argmax(s), s.shape)
            out["point_hit"] = float(B[max(iy - POINT_TOL, 0):iy + POINT_TOL + 1, max(ix - POINT_TOL, 0):ix + POINT_TOL + 1].any())
            ds = cv2.resize(s, (112, 112), interpolation=cv2.INTER_AREA)
            db = cv2.resize(B.astype(np.uint8), (112, 112), interpolation=cv2.INTER_NEAREST)
            out["box_auroc"] = float(roc_auc_score(db.ravel(), ds.ravel())) if 0 < db.sum() < db.size else np.nan
            Bd = cv2.dilate(B.astype(np.uint8), np.ones((2 * POINT_TOL + 1,) * 2, np.uint8)).astype(bool)
            out["point_chance"] = float(Bd.mean())
    return out

## 4. Part A — maps and metrics on seed-42 models

In [ ]:
AGG = {}
def run_grounding(arch, arm, seed, sample_dict, methods, tag):
    fn = os.path.join(OUT, "grounding", f"{tag}__{arch}__{arm}__s{seed}.csv")
    if os.path.exists(fn): return
    model = load_model(arch, arm, seed)
    if model is None: print("  missing checkpoint", arch, arm, seed); return
    rows = []
    for c, g in sample_dict.items():
        imgs, lungs, boxes = cache[c][arm], cache[c]["m_" + arm], cache[c]["b_" + arm]
        for _, r in g.iterrows():
            k = int(r.row); img = imgs[k]; box = boxes[k] if r.n_boxes > 0 else None
            maps = {}
            if "occlusion" in methods:
                maps["occlusion"], pred, p1 = occlusion_map(model, img)
            if "gradcam" in methods:
                maps["gradcam"], pred = gradcam(model, arch, img)
                if "occlusion" not in methods: p1 = float(probs(model, img[None])[0, 1])
            if "rollout" in methods and arch.startswith("vit"):
                rm = rollout(model, img)
                if rm is not None: maps["rollout"] = rm
            L_k = lung_def(arm, lungs[k]).astype(bool)
            ch_lung = float(L_k.mean())
            ch_les = float((box.astype(bool) & L_k).sum() / max(L_k.sum(), 1)) if box is not None else np.nan
            for meth, sal in maps.items():
                if tag == "A":
                    key = (arch, arm, c, meth, int(r.label)); a = AGG.setdefault(key, [np.zeros((S, S)), 0])
                    a[0] += norm01(sal); a[1] += 1
                for tau in TAUS:
                    rows.append(dict(arch=arch, arm=arm, seed=seed, cohort=c, image_id=r.image_id, label=int(r.label),
                                     pred=int(pred), p_pneu=p1, method=meth, tau=tau, lung_chance=ch_lung,
                                     lesion_in_lung_chance=ch_les, **metrics(sal, L_k, box, tau)))
    pd.DataFrame(rows).to_csv(fn, index=False)
    del model
    if device.type == "cuda": torch.cuda.empty_cache()
    print(f"  {tag} {arch} {arm} s{seed}: {len(rows)} rows  ({(time.time() - T0) / 60:.1f} min)")

for arch in ARCHS:
    for arm in ARMS:
        run_grounding(arch, arm, 42, GSAMPLE, ("occlusion", "gradcam", "rollout"), "A")
if AGG:
    np.savez_compressed(os.path.join(OUT, "maps", "aggregate_maps.npz"),
                        **{"|".join(map(str, k)): v[0] / max(v[1], 1) for k, v in AGG.items()})

## 5. Part B — Grad-CAM metrics for all 36 models (seed variability)

In [ ]:
for arch in ARCHS:
    for arm in ARMS:
        for seed in SEEDS:
            run_grounding(arch, arm, seed, SEEDVAR, ("gradcam",), "B")

## 6. Part C — causal region occlusion (v1 regions + lesion box), seed-42 models
Lesion box: positives occlude their radiologist box; negatives occlude a box whose size is drawn from the positive
boxes of the same cohort and arm, placed at random inside the lung (symmetric perturbation, design §6).
Lesion control: a same-size box at a random in-lung position not overlapping the lesion box.

In [ ]:
CORNER_F, BORDER_F, SUBDIA_F, UPPER_F, CENTER_F = 0.22, 0.08, 0.28, 0.20, 0.20     # v1 values (V9)
def geom_masks(Sz):
    m = {}; c = int(round(Sz * CORNER_F)); a = np.zeros((Sz, Sz), bool)
    a[:c, :c] = a[:c, -c:] = a[-c:, :c] = a[-c:, -c:] = True; m["corners"] = a
    b = int(round(Sz * BORDER_F)); a = np.zeros((Sz, Sz), bool); a[:b, :] = a[-b:, :] = a[:, :b] = a[:, -b:] = True; m["border"] = a
    a = np.zeros((Sz, Sz), bool); a[int(round(Sz * (1 - SUBDIA_F))):, :] = True; m["subdiaphragm"] = a
    a = np.zeros((Sz, Sz), bool); a[:int(round(Sz * UPPER_F)), :] = True; m["upper"] = a
    a = np.zeros((Sz, Sz), bool); lo, hi = int(round(Sz * (0.5 - CENTER_F / 2))), int(round(Sz * (0.5 + CENTER_F / 2))); a[:, lo:hi] = True
    m["center"] = a
    return m
GEOM = geom_masks(S)

def random_rect_mask(Sz, area_frac, rng):                      # v1 function (V9)
    area_frac = float(np.clip(area_frac, 0.01, 0.95)); target = area_frac * Sz * Sz
    ar = np.exp(rng.uniform(np.log(0.5), np.log(2.0)))
    h = int(np.clip(round(np.sqrt(target / ar)), 1, Sz)); w = int(np.clip(round(target / max(h, 1)), 1, Sz))
    y = rng.integers(0, Sz - h + 1); x = rng.integers(0, Sz - w + 1)
    a = np.zeros((Sz, Sz), bool); a[y:y + h, x:x + w] = True; return a

def bbox_of(m):
    ys, xs = np.where(m); return (ys.min(), ys.max() + 1, xs.min(), xs.max() + 1) if len(ys) else None

def place_in_lung(h, w, lung, avoid, rng, tries=200):
    """Random h×w rectangle whose centre lies in the lung and which does not overlap `avoid`."""
    ys, xs = np.where(lung)
    for _ in range(tries):
        if not len(ys): break
        j = rng.integers(len(ys)); y0 = int(np.clip(ys[j] - h // 2, 0, S - h)); x0 = int(np.clip(xs[j] - w // 2, 0, S - w))
        a = np.zeros((S, S), bool); a[y0:y0 + h, x0:x0 + w] = True
        if avoid is None or not (a & avoid).any(): return a
    return None

COND = ["baseline", "lungs", "non_lungs", "corners", "border", "subdiaphragm", "upper", "center",
        "rand_lungs", "rand_20", "rand_30", "lesion", "lesion_ctrl"]

def occlusion_variants(img, lung, box, size_pool, rng):
    L = lung.astype(bool)
    masks = {"baseline": np.zeros((S, S), bool), "lungs": L, "non_lungs": ~L, **GEOM,
             "rand_lungs": random_rect_mask(S, max(L.mean(), 0.02), rng),
             "rand_20": random_rect_mask(S, 0.20, rng), "rand_30": random_rect_mask(S, 0.30, rng)}
    les = box.astype(bool) if box is not None and box.any() else None
    if les is None and size_pool:                               # negative (or box-less positive): sampled size in lung
        h, w = size_pool[rng.integers(len(size_pool))]; les = place_in_lung(h, w, L, None, rng)
    ctrl = None
    if les is not None:
        bb = bbox_of(les); ctrl = place_in_lung(bb[1] - bb[0], bb[3] - bb[2], L, les, rng)
    masks["lesion"] = les if les is not None else np.zeros((S, S), bool)
    masks["lesion_ctrl"] = ctrl if ctrl is not None else np.zeros((S, S), bool)
    out = []
    for c in COND:
        v = img.copy(); v[masks[c]] = FILL; out.append(v)
    return out, {c: float(masks[c].mean()) for c in COND}, les is not None and ctrl is not None

for arch in ARCHS:
    for arm in ARMS:
        fn = os.path.join(OUT, "occlusion", f"occ__{arch}__{arm}__s42.npz")
        if os.path.exists(fn): continue
        model = load_model(arch, arm, 42)
        if model is None: continue
        blob = {}
        for c, d in EVAL.items():
            rng = np.random.default_rng(SEED)                   # same random regions for every model
            imgs, lungs, boxes = cache[c][arm], cache[c]["m_" + arm], cache[c]["b_" + arm]
            pool = []
            for k in d[(d.label == 1) & (d.n_boxes > 0)].row:
                bb = bbox_of(boxes[int(k)].astype(bool))
                if bb: pool.append((bb[1] - bb[0], bb[3] - bb[2]))
            P = np.zeros((len(d), len(COND)), np.float32); A = np.zeros_like(P); ok = np.zeros(len(d), bool)
            for i, k in enumerate(d.row.astype(int)):
                v, area, ok[i] = occlusion_variants(imgs[k], lung_def(arm, lungs[k]), boxes[k] if d.iloc[i].n_boxes > 0 else None, pool, rng)
                P[i] = probs(model, np.stack(v))[:, 1]; A[i] = [area[cc] for cc in COND]
            blob[f"{c}__p"] = P; blob[f"{c}__area"] = A; blob[f"{c}__lesion_ok"] = ok
            blob[f"{c}__y"] = d.label.values.astype(np.int8); blob[f"{c}__id"] = d.image_id.astype(str).values
            y = d.label.values
            if len(np.unique(y)) == 2:
                msg = " ".join(f"{cc}={roc_auc_score(y, P[:, j]):.3f}" for j, cc in enumerate(COND)
                               if cc in ("baseline", "lungs", "non_lungs", "lesion", "lesion_ctrl"))
                print(f"  occ {arch} {arm} {c}: {msg}")
        blob["conditions"] = np.array(COND)
        np.savez_compressed(fn, **blob)
        del model
        if device.type == "cuda": torch.cuda.empty_cache()

## 7. Consolidate, consistency checks, summary figure

In [ ]:
G = pd.concat([pd.read_csv(f) for f in glob.glob(os.path.join(OUT, "grounding", "*.csv"))], ignore_index=True)
G.to_csv(os.path.join(OUT, "grounding_per_image.csv"), index=False)
for col in ("lefr", "lefr_in_lung", "point_hit", "box_auroc", "point_chance"):
    if col not in G.columns: G[col] = np.nan
summ = (G[G.tau == 0.2].groupby(["arch", "arm", "cohort", "method"])
        .agg(n=("lfr", "size"), lfr=("lfr", "median"), lefr_in_lung=("lefr_in_lung", "median"),
             point_hit=("point_hit", "mean"), box_auroc=("box_auroc", "median")).reset_index())
ch_img = (G[G.tau == 0.2].groupby(["arch", "arm", "cohort", "method"])
          .agg(lfr_chance=("lung_chance", "mean"), lefr_in_lung_chance=("lesion_in_lung_chance", "median")).reset_index())
summ = summ.merge(ch_img, on=["arch", "arm", "cohort", "method"], how="left")       # chance under the common lung definition
summ["lfr_x_chance"] = summ.lfr / summ.lfr_chance; summ["lefr_in_lung_x_chance"] = summ.lefr_in_lung / summ.lefr_in_lung_chance
summ.to_csv(os.path.join(OUT, "grounding_summary.csv"), index=False)
print(summ[summ.method == "occlusion"].round(3).to_string(index=False))

rows = []
for f in sorted(glob.glob(os.path.join(OUT, "occlusion", "*.npz"))):
    z = np.load(f, allow_pickle=False); _, arch, arm, _ = os.path.basename(f)[:-4].split("__")
    cond = [str(x) for x in z["conditions"]]
    for c in EVAL:
        if f"{c}__p" not in z.files: continue
        y, P = z[f"{c}__y"], z[f"{c}__p"]
        if len(np.unique(y)) < 2: continue
        base = roc_auc_score(y, P[:, 0])
        for j, cc in enumerate(cond):
            a = roc_auc_score(y, P[:, j])
            rows.append(dict(arch=arch, arm=arm, cohort=c, condition=cc, auc=a, d_auc=base - a,
                             area=float(z[f"{c}__area"][:, j].mean())))
O = pd.DataFrame(rows)
ctrl_of = {"lungs": "rand_lungs", "corners": "rand_20", "center": "rand_20", "upper": "rand_20", "border": "rand_30",
           "subdiaphragm": "rand_30", "lesion": "lesion_ctrl"}
if len(O):
    key = O.set_index(["arch", "arm", "cohort", "condition"]).d_auc
    O["net_effect"] = [r.d_auc - key.get((r.arch, r.arm, r.cohort, ctrl_of[r.condition]), np.nan)
                       if r.condition in ctrl_of else np.nan for r in O.itertuples()]
O.to_csv(os.path.join(OUT, "occlusion_v2_summary.csv"), index=False)
for arch in (O.arch.unique() if len(O) else []):              # v1 consistency checks, now per architecture
    for c in O.cohort.unique():
        q = O[(O.arch == arch) & (O.arm == "lung") & (O.cohort == c)].set_index("condition").d_auc
        if len(q):
            print(f"  check {arch:<30} {c:<8} C/non_lungs dAUC={q.get('non_lungs', np.nan):+.3f} (expect ~0)  "
                  f"C/lungs dAUC={q.get('lungs', np.nan):+.3f} (expect large)")

fig, axs = plt.subplots(1, 2, figsize=(7.16, 2.6), layout="constrained")
s0 = summ[(summ.method == "occlusion")]
for ax, (col, ttl) in zip(axs, (("lfr_x_chance", "Lung focus (× chance)"), ("lefr_in_lung_x_chance", "Lesion focus within lung (× chance)"))):
    piv = s0.groupby(["arch", "arm"])[col].median().unstack()
    if piv.empty or piv.isna().all().all(): ax.set_axis_off(); continue
    piv = piv.reindex(columns=[a for a in ARMS if a in piv.columns])
    ax.imshow(piv.values, cmap="viridis", aspect="auto")
    ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels(piv.columns); ax.set_yticks(range(piv.shape[0]))
    ax.set_yticklabels([a.split("_")[0] for a in piv.index]); ax.set_title(ttl, loc="left", fontsize=8)
    vmax = np.nanmax(piv.values)
    for (i, j), v in np.ndenumerate(piv.values):
        ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=6, color="white" if v < vmax * 0.6 else "black")
fig.savefig(os.path.join(OUT, "fig_v14_grounding_overview.png"), dpi=300); plt.close(fig)
print(f"V14 complete in {(time.time() - T0) / 3600:.2f} h")